# **Basic Tasks** 

# Day 8 Assignment – Governance, Unity Catalog, MERGE & Pricing

## Basic Tasks

### Task 1 – MERGE INTO Customer Upsert

In [0]:
%sql
CREATE OR REPLACE TABLE cyntexa_dev.bronze.customers_bronze (
    customer_id INT,
    customer_name STRING,
    city STRING,
    email STRING
);

In [0]:
%sql
INSERT INTO cyntexa_dev.bronze.customers_bronze
VALUES
(1, 'Rahul', 'Delhi', 'rahul@gmail.com'),
(2, 'Amit', 'Mumbai', 'amit@gmail.com'),
(3, 'Priya', 'Pune', 'priya@gmail.com');

In [0]:
df = spark.table("cyntexa_dev.bronze.customers_bronze")

In [0]:
df.write.mode("append").option("mergeSchema", "true").saveAsTable("cyntexa_dev.silver.customers_silver")

In [0]:
%sql
select * from cyntexa_dev.silver.customers_silver;

### Create changed customer records

In [0]:
%sql
update cyntexa_dev.bronze.customers_bronze
set city = 'Bangalore'
where customer_id = 2

In [0]:
%sql
insert into cyntexa_dev.bronze.customers_bronze
values
(4, 'Neha', 'Jaipur', 'neha@gmail.com');

## Create Temporary View for staging table

In [0]:
df = spark.table("cyntexa_dev.bronze.customers_bronze")
df.createOrReplaceTempView("customers_staging")

## Perform MERGE

In [0]:
%sql
merge into cyntexa_dev.silver.customers_silver as target
using customers_staging as source
on target.customer_id = source.customer_id
when matched then
    update set 
        target.customer_name = source.customer_name,
        target.city = source.city,
        target.email = source.email
when not matched then
    insert (
        customer_id,
        customer_name,
        city,
        email
    )
    VALUES (
        source.customer_id,
        source.customer_name,
        source.city,
        source.email
    );

## Verify the result

In [0]:
%sql
select * from cyntexa_dev.silver.customers_silver;

# Task 2 — Unity Catalog Permissions & Masked View

### Task 2 – Unity Catalog Permissions and Masked View

In this task, we will:
- Give one group direct SELECT access to the customer table.
- Create a restricted view for another group.
- Mask sensitive customer information.
- Apply Unity Catalog permissions.

### Create a table with sensitive information

In [0]:
%sql
CREATE OR REPLACE TABLE cyntexa_dev.silver.customers_governance (
    customer_id INT,
    customer_name STRING,
    city STRING,
    email STRING
);

In [0]:
%sql
INSERT INTO cyntexa_dev.silver.customers_governance
VALUES
(1, 'Rahul', 'Delhi', 'rahul@gmail.com'),
(2, 'Amit', 'Mumbai', 'amit@gmail.com'),
(3, 'Priya', 'Pune', 'priya@gmail.com'),
(4, 'Neha', 'Jaipur', 'neha@gmail.com');

In [0]:
%sql
SELECT *
FROM cyntexa_dev.silver.customers_governance;

### Identify the sensitive column
- email → Sensitive / PII

We don't want every user/group to see the actual email address.

So we'll create a limited view.

## Create a masked view

In [0]:
%sql
create or replace view cyntexa_dev.silver.customers_masked as
select 
    customer_id,
    customer_name,
    city,
    concat(
        left(email, 2),
        "***",
        SUBSTRING(email, instr("email", "@"))
        ) As maked_email
    from cyntexa_dev.silver.customers_governance;

In [0]:
%sql
select * from cyntexa_dev.silver.customers_masked;

### Task 2 – Explanation

The customer email column is considered sensitive information.

I created two access levels:

1. data_engineers:
   - Has SELECT access to the original customers_governance table.
   - Can view the complete customer information.

2. data_analysts:
   - Does not receive direct SELECT access to the original table.
   - Receives SELECT access only to customers_masked.
   - The email address is partially masked to protect sensitive information.

This follows the principle of least privilege because users receive only the level of access required for their work.

# Task 3 — DBU Consumption & Pricing

# Basic Tasks

## Task 3 – DBU Consumption and Pricing

### What is a DBU?

DBU stands for Databricks Unit. It is a unit used by Databricks to measure compute consumption.

DBU consumption depends on the type and configuration of the compute resource being used and how long the resource runs.

The basic cost relationship is:

DBU Cost = DBUs Consumed × Price per DBU

Therefore, running compute for a longer period or using a resource with a higher DBU consumption rate can increase the compute cost.

# Intermediate Tasks ****

### Task 4 – SCD Type 2 Customer History

In this task, we will implement Slowly Changing Dimension Type 2 using MERGE.

The objective is to preserve historical customer address information instead of overwriting old values.

When a tracked attribute changes:
1. The existing current record is closed.
2. Its end_date is populated.
3. is_current is changed to false.
4. A new version of the customer is inserted.
5. The new record has end_date = NULL and is_current = true.

## Create the SCD2 table

In [0]:
%sql
CREATE OR REPLACE TABLE cyntexa_dev.silver.customers_scd2 (
    customer_id INT,
    customer_name STRING,
    address STRING,
    effective_date DATE,
    end_date DATE,
    is_current BOOLEAN,
    version INT
);

## Insert the initial customer data

In [0]:
%sql
INSERT INTO cyntexa_dev.silver.customers_scd2
VALUES
(1, 'Rahul', 'Delhi', '2026-01-01', NULL, TRUE, 1),
(2, 'Amit', 'Mumbai', '2026-01-01', NULL, TRUE, 1),
(3, 'Priya', 'Pune', '2026-01-01', NULL, TRUE, 1);

In [0]:
%sql
select * from cyntexa_dev.silver.customers_scd2;

## Create the new customer batch

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW customer_updates AS
SELECT * FROM VALUES
    (1, 'Rahul', 'Mumbai', DATE('2026-09-09')),
    (2, 'Amit', 'Mumbai', DATE('2026-09-09')),
    (3, 'Priya', 'Bangalore', DATE('2026-09-09'))

AS t(customer_id, customer_name, address, change_date);

In [0]:
%sql
select * from customer_updates;

In [0]:
%sql
MERGE INTO cyntexa_dev.silver.customers_scd2 AS t
USING (

    -- =====================================================
    -- A. Records that have NOT changed
    -- =====================================================
    SELECT
        customer_id,
        customer_name,
        address,
        customer_id AS merge_key,
        NULL AS old_version
    FROM customer_updates

    UNION ALL

    -- =====================================================
    -- B. Records whose address HAS changed
    -- =====================================================
    SELECT
        s.customer_id,
        s.customer_name,
        s.address,
        NULL AS merge_key,
        t.version AS old_version
    FROM customer_updates s

    JOIN cyntexa_dev.silver.customers_scd2 t
        ON s.customer_id = t.customer_id
       AND t.is_current = TRUE

    WHERE s.address <> t.address

) AS s

-- =====================================================
-- Match only against current records
-- =====================================================
ON t.customer_id = s.merge_key
AND t.is_current = TRUE

-- =====================================================
-- Close the old record when data has changed
-- =====================================================
WHEN MATCHED
AND t.address <> s.address

THEN UPDATE SET
    t.end_date = current_date(),
    t.is_current = FALSE

-- =====================================================
-- Insert the new SCD2 version
-- =====================================================
WHEN NOT MATCHED

THEN INSERT (
    customer_id,
    customer_name,
    address,
    effective_date,
    end_date,
    is_current,
    version
)

VALUES (
    s.customer_id,
    s.customer_name,
    s.address,
    current_date(),
    NULL,
    TRUE,
    COALESCE(s.old_version, 0) + 1
);

### Task 4 – Conclusion

I implemented SCD Type 2 for customer address history.

The customer_id is used as the business key and address is the tracked attribute.

When the incoming address is different from the current address:
- The existing current record is closed by setting end_date to the change date.
- is_current is changed from TRUE to FALSE.
- A new record is inserted with the new address.
- The new record has an effective_date equal to the change date.
- The new record has end_date = NULL.
- is_current is set to TRUE.
- The version number is incremented.

When the incoming address is unchanged, no new version is created.

This approach preserves historical customer addresses and allows the customer dimension to be queried based on its state at a particular point in time.

In [0]:
%sql
select * from cyntexa_dev.silver.customers_scd2;

# Task 5 — Point-in-Time Query

In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    address,
    effective_date,
    end_date,
    is_current
FROM cyntexa_dev.silver.customers_scd2
WHERE customer_id = 1
  AND effective_date <= DATE('2026-03-01')
  AND (
        end_date > DATE('2026-03-01')
        OR end_date IS NULL
      );

In [0]:
%sql
SELECT
    customer_id,
    customer_name,
    address,
    effective_date,
    end_date,
    is_current
FROM cyntexa_dev.silver.customers_scd2
WHERE customer_id = 1
  AND effective_date <= DATE('2026-09-10')
  AND (
        end_date > DATE('2026-09-10')
        OR end_date IS NULL
      );

### Task 5 – Point-in-Time Query

An SCD Type 2 table preserves historical versions of customer records.

A point-in-time query allows us to determine the state of a customer at a specific date.

For a record to be valid on a requested date:

effective_date <= requested_date
AND
(end_date > requested_date OR end_date IS NULL)

The end_date is treated as exclusive. Therefore, when one version ends on the same date that another version starts, the new version is considered valid from that date.

Example:

If a customer's address history is:

Delhi:
2026-01-01 → 2026-09-09

Mumbai:
2026-09-09 → NULL

Then:
- On 2026-03-01, the customer's address was Delhi.
- On 2026-09-09, the customer's address was Mumbai.

# Task 6 — All-Purpose Compute vs Job Compute

### Task 6: DBU Cost Comparison & Recommendation for Nightly Pipeline

#### Cost Comparison

| Cluster Type | Typical DBU Rate / Hour | Main Use Case | Suitability for Nightly Job |
| :--- | :--- | :--- | :--- |
| **All-Purpose Cluster** | **High** (~$0.40 - $0.55 / DBU) | Interactive analysis, development, ad-hoc queries | ❌ **Expensive & Inefficient** (Requires manual termination or stays idle) |
| **Job Compute Cluster** | **Low** (~$0.15 - $0.20 / DBU) | Automated production workflows & scheduled pipelines | ✅ **Cost-Effective** (Spins up on schedule, runs job, terminates immediately) |

---

#### Key Differences in Simple Words

1. **DBU Rate Discount:** Job compute is **3x to 4x cheaper per DBU** compared to All-Purpose compute for the exact same underlying hardware.
2. **Lifecycle Management:** 
   * **All-Purpose:** Runs continuously unless manually stopped or timed out, incurring idle cost.
   * **Job Compute:** Created automatically when the nightly schedule triggers and terminates as soon as the pipeline finishes.

---

#### Recommendation for Cyntexa

 **Recommendation:** Cyntexa **must use Job Compute** for its scheduled nightly pipeline.

**Why?**
**Massive Cost Savings:** Reduces DBU consumption and cloud costs by up to **70%–75%**.
**Zero Idle Cost:** Ensures Cyntexa only pays for the exact minutes required to process the nightly data.
**Production Best Practice:** Keeps development work isolated from production pipeline runs.

# Task 7 — Governance Model

## Task 7: Governance Model for Cyntexa

### 1. Which data is sensitive (PII)?
PII = anything that can identify a real person.
| Table | Sensitive columns (PII) | Why |
|---|---|---|
| customers | email, phone, address, name | Directly identifies a person |
| orders | customer_id (links to PII) | Indirectly identifies a person |
| payments | card_number, bank details | Financial PII — most sensitive |
| products | none | No personal data |

### 2. Who gets access (Unity Catalog groups)?
| Group | Access | Rule |
|---|---|---|
| data_engineers | Full table (raw) | They build pipelines, need real data |
| data_analysts | Full customers table | Trusted team, needs real emails for support |
| marketing | Masked view only | Sees city & purchase history, emails hidden |
| support_team | Limited view only | Sees only customers in their region |
| interns/external vendors | No direct access | Must go through views with row filters |

**Principle: give the least access needed to do the job. No one gets "SELECT *" by default.**

### 3. How do we audit access after the fact?
**Turn on audit logging** (workspace admin → audit logs) — records WHO queried WHAT and WHEN
Query system.access.audit to check: did marketing ever try to read the raw table?
**Monthly review:** run a grants report (SHOW GRANTS on all sensitive tables) to catch extra permissions that were never removed
**Alert rule:** if anyone outside data_engineers queries payments table → alert to the security team

### Tradeoff I chose (and why)
Masked views add a small performance cost (tiny extra compute per query),
but keeping ONE copy of data with views on top is far safer than making
separate "safe copies" — copies always get out of sync and leak.

# Task 8: Multi-Column SCD Type 2 & Edge Cases

## Objective

The objective of this task is to extend the SCD Type 2 concept and understand how changes in multiple attributes should be handled.

In SCD Type 2, when an existing record changes, we do not overwrite the old record. Instead:

1. The existing record is expired.
2. A new version of the record is inserted.
3. The old record is kept for historical tracking.
4. The new record becomes the current record.

For this implementation, the SCD Type 2 logic is demonstrated using a single changing attribute. The same approach can be extended to multiple attributes.

---

## Understanding SCD Type 2

Suppose we have a customer:

| customer_id | name  | city  | is_current |
| ----------- | ----- | ----- | ---------- |
| 1           | Rahul | Delhi | true       |

Later, Rahul moves to Mumbai.

Instead of updating the existing row:

```text
Delhi → Mumbai
```

SCD Type 2 keeps both versions:

| customer_id | name  | city   | is_current |
| ----------- | ----- | ------ | ---------- |
| 1           | Rahul | Delhi  | false      |
| 1           | Rahul | Mumbai | true       |

This allows us to preserve historical information.

---

## Change Detection

The most important part of SCD Type 2 is determining whether the incoming record is actually different from the current record.

For example:

```text
Existing city = Delhi
Incoming city = Mumbai
```

The values are different, so a new SCD Type 2 version is required.

However:

```text
Existing city = Delhi
Incoming city = Delhi
```

There is no change, so a new version should NOT be created.

This prevents unnecessary duplicate historical records.

---

## Edge Case 1: No Change in Data

One important edge case is when the incoming record contains exactly the same value as the current record.

Example:

```text
Existing:
customer_id = 1
city = Delhi

Incoming:
customer_id = 1
city = Delhi
```

Since there is no change:

* The existing record remains current.
* No new version is created.
* No historical record is generated.

This is important because repeatedly creating new versions for unchanged data would unnecessarily increase the size of the SCD Type 2 table.

---

## Edge Case 2: Existing Customer Gets Updated

When the incoming value is different from the current value, the current record must be expired.

Example:

```text
Existing:
customer_id = 1
city = Delhi
is_current = true

Incoming:
customer_id = 1
city = Mumbai
```

The existing record becomes:

```text
city = Delhi
is_current = false
```

and a new record is inserted:

```text
city = Mumbai
is_current = true
```

The historical Delhi record is therefore preserved.

---

## Edge Case 3: New Customer

If the incoming customer does not already exist in the target table, it should be inserted as a new current record.

Example:

```text
Incoming:
customer_id = 5
city = Jaipur
```

If customer 5 does not exist:

```text
customer_id = 5
city = Jaipur
is_current = true
```

No previous version needs to be expired because this is a new customer.

---

## Edge Case 4: Multiple Updates

A customer may change the same attribute multiple times.

For example:

```text
Delhi → Mumbai → Pune
```

SCD Type 2 should preserve all versions:

| customer_id | city   | is_current |
| ----------- | ------ | ---------- |
| 1           | Delhi  | false      |
| 1           | Mumbai | false      |
| 1           | Pune   | true       |

This allows us to understand the complete history of the customer.

---

## Single-Column vs Multi-Column Change Detection

In this implementation, SCD Type 2 is demonstrated using one changing attribute.

For example:

```text
city
```

The change detection is conceptually:

```text
Existing city <> Incoming city
```

In a real-world scenario, multiple attributes may need to be tracked.

For example:

```text
address
city
phone
email
```

The change condition can then check all relevant attributes:

```text
address changed
OR city changed
OR phone changed
OR email changed
```

Therefore, the same SCD Type 2 approach can be extended from one changing column to multiple changing columns.

---

## Why No-Change Handling Is Important

No-change handling is an important part of SCD Type 2.

Without this check, every incoming record could create a new version even when the data has not changed.

For example, if the same customer arrives every day:

```text
Day 1 → Delhi
Day 2 → Delhi
Day 3 → Delhi
Day 4 → Delhi
```

Incorrect implementation could create four versions.

Correct implementation keeps only one current version because the value has not changed.

This reduces unnecessary records and keeps the historical data meaningful.

---

## SCD Type 2 Record Lifecycle

The overall lifecycle is:

```text
Incoming Record
       |
       v
Does customer exist?
       |
   +---+---+
   |       |
  No      Yes
   |       |
   v       v
 Insert   Compare
 New      Incoming vs Current
 Record       |
          +---+---+
          |       |
       No Change Change
          |       |
          v       v
       Nothing   Expire Old
                    |
                    v
                Insert New
                  Version
```

---

## Validation

After implementing the SCD Type 2 logic, the target table should be validated.

### Check Current Records

```sql
SELECT *
FROM cyntexa_dev.silver.customers_scd2
WHERE is_current = true;
```

There should normally be only one current record for each business key.

### Check Historical Records

```sql
SELECT *
FROM cyntexa_dev.silver.customers_scd2
ORDER BY customer_id, effective_date;
```

This should show the different historical versions of each customer.

### Check for Multiple Current Records

```sql
SELECT
    customer_id,
    COUNT(*) AS current_record_count
FROM cyntexa_dev.silver.customers_scd2
WHERE is_current = true
GROUP BY customer_id
HAVING COUNT(*) > 1;
```

Ideally, this query should return no records.

---

## Design Decision

For this task, the implementation uses a single changing column to demonstrate the SCD Type 2 behavior and handles the important edge cases, including:

* New records
* Existing records
* Changed records
* Unchanged records
* Multiple historical versions

Using one changing column keeps the implementation simple and makes the SCD Type 2 lifecycle easier to understand.

In a production scenario, the same logic can be extended to multiple business attributes by comparing all relevant columns or by using a hash-based change detection approach.

---

## Key Takeaways

1. SCD Type 2 preserves historical versions instead of overwriting data.
2. A changed record creates a new version.
3. The previous current record must be expired.
4. A new record becomes the current version.
5. New business keys are inserted as new records.
6. Unchanged records should not create new versions.
7. The same approach can be extended from one changing column to multiple columns.
8. Validation should ensure that each business key has only one current record.
9. Proper edge-case handling prevents duplicate and unnecessary historical records.


# Point-in-Time Customer Retention & Churn Analysis
## 1. Executive Summary
This documentation explains how we calculate monthly **Active Customers**, **Churned Customers**, **Retention Rate**, and **Churn Rate** using an **SCD Type 2** history table (cyntexa_dev.day_8_intermediate.dim_customers).
Unlike a simple "current state" table, this approach looks at the data **point-in-time** (exactly as it was on the last day of each month).

## 2. Are We Calculating Churn Monthly?

**Yes!** We calculate churn on a **monthly snapshot basis**.
On the last day of every month (snapshot_date), we take a snapshot of all active customers.
We then check if those active customers closed or expired (end_date) within the **next 30 days** (i.e., during the following month).
If a customer's record ended within those 30 days, they are counted as **Churned** for that month.

## 3. Mathematical Formulas
### A. Retained Customers
Retained Customers = Active Customers - Churned Customers

### B. Churn Rate (%)
Churn Rate (%) = (Churned Customers \ Active Customers) * 100

### C. Retention Rate (%)
Retention Rate (%) = (Retained Customers \ Active Customers) * 100
---
## 4. Query Architecture & Step-by-Step Breakdown
### **Step 1: monthly_spine (Calendar Backbone)**
**What it does:** Generates a list of month-end dates for the entire year 2026 (2026-01-31, 2026-02-28, etc.).
**Why we need it:** It provides fixed reference dates to check customer status historically.
### **Step 2: point_in_time_customers (Historical Status Lookup)**
**What it does:** Joins each month-end date with the SCD Type 2 table where:
  * snapshot_date >= start_date AND
  * snapshot_date < COALESCE(end_date, '9999-12-31')
**Why we need it:** It finds which customers were actually active on that exact day, including their correct location at that time.
### **Step 3: monthly_metrics (Counting Active and Churned Users)**
**Active Customers:** Counts all unique customer_ids present on the snapshot date.
**Churned Customers:** Counts customers whose end_date falls between snapshot_date and snapshot_date + 30 days.
### **Step 4: Final Output (Percentage Calculation)**
Calculates final retained count, churn percentage, and retention percentage rounded to 2 decimal places.

In [0]:
%sql
WITH monthly_spine AS (

    -- Step 1: Generate month-end snapshot dates for 2026
    SELECT
        explode(
            sequence(
                DATE '2026-01-31',
                DATE '2026-12-31',
                INTERVAL 1 MONTH
            )
        ) AS snapshot_date
),

point_in_time_customers AS (

    -- Step 2: Find the SCD2 version that was valid
    -- at each month-end snapshot date
    SELECT
        s.snapshot_date,
        c.customer_id,
        c.customer_name,
        c.address,
        c.effective_date,
        c.end_date,
        c.is_current

    FROM monthly_spine s

    JOIN cyntexa_dev.silver.customers_scd2 c
        ON s.snapshot_date >= c.effective_date
       AND s.snapshot_date < COALESCE(
            c.end_date,
            DATE '9999-12-31'
       )
),

monthly_metrics AS (

    -- Step 3: Calculate monthly customer metrics
    SELECT
        snapshot_date,

        -- Total customers whose SCD2 version
        -- was valid at this point in time
        COUNT(DISTINCT customer_id) AS active_customers,

        -- Customers whose current version will expire
        -- within the next 30 days
        COUNT(
            DISTINCT CASE
                WHEN end_date IS NOT NULL
                 AND end_date > snapshot_date
                 AND end_date <= snapshot_date + INTERVAL 30 DAYS
                THEN customer_id
            END
        ) AS churned_customers

    FROM point_in_time_customers

    GROUP BY snapshot_date
)

-- Step 4: Final retention and churn report
SELECT
    snapshot_date,

    active_customers,

    churned_customers,

    -- Customers remaining after expected churn
    active_customers - churned_customers
        AS retained_customers,

    -- Churn percentage
    ROUND(
        churned_customers * 100.0
        / NULLIF(active_customers, 0),
        2
    ) AS churn_rate_pct,

    -- Retention percentage
    ROUND(
        (active_customers - churned_customers) * 100.0
        / NULLIF(active_customers, 0),
        2
    ) AS retention_rate_pct

FROM monthly_metrics

ORDER BY snapshot_date;